# 폐암 데이터 생존분석 실습 — 통합 분석 노트북
**대한폐암학회 표적치료연구회 · 생성형AI 의료데이터 분석 워크숍 (LAB 1–2 모범답안 코드)**

이 노트북은 다음 분석을 순서대로 수행합니다.
1. 데이터 구조·품질 점검 (결측치, 이상치, 논리 검사)
2. 생존군 비교 Table 1 (정규성 검정 기반 t-검정/Mann-Whitney U, 카이제곱)
3. Kaplan-Meier 생존분석 — 전체 코호트, 병기별 (number at risk, 95% CI, log-rank)
4. 유전자 변이 중심 분석 — 변이별 KM, 표적치료 수혜율, EGFR 하위그룹
5. 다변량 Cox 비례위험 모형 — Schoenfeld 잔차 검정, 층화 민감도분석, forest plot

> ⚠️ 실습 데이터는 공개 교육용 **합성 데이터**입니다. 임상적 결론 도출에 사용할 수 없습니다.

**사용법**: 위에서부터 셀을 순서대로 실행(Shift+Enter)하고, STEP 0-2에서 `1_WHO_lung_cancer_dataset.csv` 파일을 선택해 업로드하세요.

## STEP 0. 환경 설정 및 데이터 업로드

In [ ]:
# STEP 0-1. 라이브러리 설치 및 불러오기 (Colab 기본 환경 + lifelines)
!pip install lifelines -q

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test, pairwise_logrank_test, logrank_test, proportional_hazard_test
from lifelines.utils import median_survival_times
from lifelines.plotting import add_at_risk_counts
import os
os.makedirs('outputs', exist_ok=True)

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False})
# 색각 이상자도 구분 가능한 Okabe-Ito 팔레트
OKABE = {'blue':'#0072B2','green':'#009E73','orange':'#E69F00','vermillion':'#D55E00',
         'pink':'#CC79A7','gray':'#555555'}
print('환경 준비 완료')

In [ ]:
# STEP 0-2. 파일 업로드 (실행 후 나타나는 버튼으로 CSV 선택)
from google.colab import files
uploaded = files.upload()
FNAME = list(uploaded.keys())[0]
df = pd.read_csv(FNAME)
print(f'업로드 완료: {FNAME} — {df.shape[0]}명 x {df.shape[1]}개 변수')

## STEP 1. 데이터 구조 및 품질 점검
결측치·범위·논리 일관성을 분석 전에 반드시 확인합니다. AI가 생성한 분석도 이 단계를 건너뛰면 신뢰할 수 없습니다.

In [ ]:
# STEP 1-1. 구조 요약
print('== 변수 유형 ==')
print(df.dtypes.value_counts())
print('\n== 결측치 ==')
miss = df.isna().sum()
print(miss[miss>0] if miss.sum()>0 else '결측치 없음')

print('\n== 수치형 변수 범위 ==')
for c in ['Age','BMI','Cigarettes_Per_Day','Years_Smoking','Tumor_Size_cm','Survival_Months']:
    s = df[c]
    print(f'{c}: min={s.min()}, max={s.max()}, mean={s.mean():.1f}, median={s.median()}')

In [ ]:
# STEP 1-2. 논리 일관성 검사 (이상 소견은 분석 방침 결정에 사용)
checks = {
 '비흡연자인데 흡연량>0': ((df.Smoking_Status=='Never Smoked') & (df.Cigarettes_Per_Day>0)).sum(),
 '흡연기간 >= 나이': (df.Years_Smoking >= df.Age).sum(),
 'Stage IV인데 전이 없음': ((df.Cancer_Stage=='Stage IV') & (df.Metastasis=='No')).sum(),
 'Stage I-III인데 전이 있음(주의)': ((df.Cancer_Stage.isin(['Stage I','Stage II','Stage III'])) & (df.Metastasis=='Yes')).sum(),
 '종양 12cm 초과(극단값)': (df.Tumor_Size_cm>12).sum(),
 'SCLC인데 표적치료(비현실)': ((df.Cancer_Type=='SCLC') & (df.Treatment=='Targeted Therapy')).sum(),
}
for k,v in checks.items(): print(f'{k}: {v}건')
print('\n비흡연자 비율: {:.1f}% (실제 폐암 역학과 비교해 볼 것)'.format((df.Smoking_Status=='Never Smoked').mean()*100))

## STEP 2. 생존군 비교 Table 1
정규성(Shapiro-Wilk)을 먼저 확인해 검정법을 선택합니다: 정규 → 평균±SD와 t-검정, 비정규 → 중앙값[IQR]과 Mann-Whitney U.

In [ ]:
# STEP 2-1. 정규성 검정
for c in ['Age','BMI','Tumor_Size_cm']:
    _, p_sh = stats.shapiro(df[c].sample(min(500,len(df)), random_state=42))
    print(f'{c}: skew={df[c].skew():.2f}, Shapiro-Wilk p={p_sh:.4f} -> {"정규(t-검정)" if p_sh>=0.05 else "비정규(Mann-Whitney U)"}')

In [ ]:
# STEP 2-2. Table 1 생성
g1, g0 = df[df.Survived=='Yes'], df[df.Survived=='No']
rows = []

def cont_row(label, col, normal):
    if normal:
        v1 = f'{g1[col].mean():.1f} ± {g1[col].std():.1f}'
        v0 = f'{g0[col].mean():.1f} ± {g0[col].std():.1f}'
        p = stats.ttest_ind(g1[col], g0[col]).pvalue; note='t-test'
    else:
        q = lambda g: f'{g[col].median():.1f} [{g[col].quantile(.25):.1f}–{g[col].quantile(.75):.1f}]'
        v1, v0 = q(g1), q(g0)
        p = stats.mannwhitneyu(g1[col], g0[col]).pvalue; note='Mann-Whitney U'
    rows.append([label, v1, v0, f'{p:.3f}' if p>=0.001 else '<0.001', note])

def cat_row(label, col, order=None):
    ct = pd.crosstab(df[col], df.Survived)
    p = stats.chi2_contingency(ct)[1]
    rows.append([f'{label}, n (%)', '', '', f'{p:.3f}' if p>=0.001 else '<0.001', 'Chi-square'])
    idx = order if order else ct.index
    for k in idx:
        rows.append([f'  {k}', f'{ct.loc[k,"Yes"]} ({ct.loc[k,"Yes"]/len(g1)*100:.1f})',
                     f'{ct.loc[k,"No"]} ({ct.loc[k,"No"]/len(g0)*100:.1f})', '', ''])

cont_row('Age, years, mean ± SD', 'Age', normal=True)
cont_row('BMI, kg/m², median [IQR]', 'BMI', normal=False)
cont_row('Tumor size, cm, median [IQR]', 'Tumor_Size_cm', normal=False)
cat_row('Sex', 'Gender')
cat_row('Smoking status', 'Smoking_Status', ['Never Smoked','Former Smoker','Current Smoker'])
cat_row('Cancer stage', 'Cancer_Stage', ['Stage I','Stage II','Stage III','Stage IV'])
cat_row('Histologic subtype', 'NSCLC_Subtype')
cat_row('Genetic mutation', 'Genetic_Mutation')
cat_row('Treatment', 'Treatment')
cat_row('Distant metastasis', 'Metastasis')

table1 = pd.DataFrame(rows, columns=['Characteristic', f'Survivors (n={len(g1)})', f'Non-survivors (n={len(g0)})', 'P value', 'Test'])
table1.to_csv('outputs/Table1.csv', index=False)
table1

## STEP 3. Kaplan-Meier 생존분석
사망(Survived='No')을 사건, 생존을 중도절단으로 정의합니다.

In [ ]:
# STEP 3-1. 전체 코호트 KM 곡선 (Figure 1)
df['event'] = (df.Survived=='No').astype(int)
T, E = df.Survival_Months, df.event

kmf = KaplanMeierFitter().fit(T, E, label=f'All patients (N = {len(df):,})')
med = kmf.median_survival_time_
ci = median_survival_times(kmf.confidence_interval_)
lo, hi = ci.iloc[0,0], ci.iloc[0,1]

fig, ax = plt.subplots(figsize=(7.2,5.6), dpi=150)
kmf.plot_survival_function(ax=ax, ci_show=True, color=OKABE['blue'], linewidth=2)
ax.axhline(0.5, color='gray', lw=0.8, ls='--', alpha=0.6)
ax.axvline(med, color='gray', lw=0.8, ls='--', alpha=0.6)
ax.annotate(f'Median OS: {med:.0f} months\n(95% CI, {lo:.0f}–{hi:.0f})', xy=(med,0.5),
            xytext=(med+12,0.62), fontsize=10.5, arrowprops=dict(arrowstyle='-', color='gray', lw=0.8))
ax.set_xlabel('Time since diagnosis (months)'); ax.set_ylabel('Overall survival probability')
ax.set_ylim(0,1.02); ax.legend(loc='upper right', frameon=False)
ax.set_title('Kaplan-Meier Estimate of Overall Survival', pad=12)
add_at_risk_counts(kmf, ax=ax, rows_to_show=['At risk'])
plt.tight_layout(); plt.savefig('outputs/Figure1_KM_overall.png', dpi=300, bbox_inches='tight'); plt.show()
print(f'중앙 생존기간: {med:.0f}개월 (95% CI {lo:.0f}–{hi:.0f}), 사건 {E.sum()}건')

In [ ]:
# STEP 3-2. 병기별 KM 곡선 + log-rank (Figure 2)
stages = ['Stage I','Stage II','Stage III','Stage IV']
scol = dict(zip(stages, [OKABE['blue'],OKABE['green'],OKABE['orange'],OKABE['vermillion']]))

res = multivariate_logrank_test(T, df.Cancer_Stage, E)
ptxt = 'Log-rank P < 0.001' if res.p_value<0.001 else f'Log-rank P = {res.p_value:.3f}'

fig, ax = plt.subplots(figsize=(7.8,6.4), dpi=150)
fitters=[]
for s in stages:
    m = df.Cancer_Stage==s
    k = KaplanMeierFitter().fit(T[m], E[m], label=f'{s} (n = {m.sum()})')
    k.plot_survival_function(ax=ax, ci_show=True, ci_alpha=0.12, color=scol[s], linewidth=2)
    fitters.append(k)
    print(f'{s}: n={m.sum()}, median={k.median_survival_time_:.0f}개월')
ax.text(0.97,0.965, ptxt, transform=ax.transAxes, ha='right', va='top', fontweight='bold')
ax.set_xlabel('Time since diagnosis (months)'); ax.set_ylabel('Overall survival probability')
ax.set_ylim(0,1.02); ax.legend(loc='lower left', frameon=False, fontsize=10)
ax.set_title('Overall Survival by Cancer Stage', pad=12)
add_at_risk_counts(*fitters, ax=ax, rows_to_show=['At risk'])
plt.tight_layout(); plt.savefig('outputs/Figure2_KM_by_stage.png', dpi=300, bbox_inches='tight'); plt.show()

print(f'\n전체 log-rank: chi2={res.test_statistic:.1f}, p={res.p_value:.2e}')
print('\n쌍별 log-rank p값:'); print(pairwise_logrank_test(T, df.Cancer_Stage, E).summary['p'].round(4))

## STEP 4. 유전자 변이 중심 분석 (표적치료 관점)
**검증 관점**: 결과를 실제 임상 근거(FLAURA, ALEX 등)와 비교하며 비현실적 패턴을 찾아보세요.

In [ ]:
# STEP 4-1. 변이별 KM 곡선 (Figure 3)
groups = ['EGFR','ALK','KRAS','ROS1','No Mutation']
sub = df[df.Genetic_Mutation.isin(groups)]
mcol = dict(zip(groups, [OKABE['blue'],OKABE['green'],OKABE['orange'],OKABE['pink'],OKABE['gray']]))

res = multivariate_logrank_test(sub.Survival_Months, sub.Genetic_Mutation, sub.event)
fig, ax = plt.subplots(figsize=(7.8,6.6), dpi=150)
fitters=[]
for g in groups:
    m = sub.Genetic_Mutation==g
    k = KaplanMeierFitter().fit(sub.Survival_Months[m], sub.event[m], label=f'{g} (n = {m.sum()})')
    k.plot_survival_function(ax=ax, ci_show=True, ci_alpha=0.10, color=mcol[g], linewidth=2,
                             linestyle='--' if g=='No Mutation' else '-')
    fitters.append(k)
    print(f'{g}: n={m.sum()}, median={k.median_survival_time_:.0f}개월')
ax.text(0.97,0.965, f'Log-rank P = {res.p_value:.3f}', transform=ax.transAxes, ha='right', va='top', fontweight='bold')
ax.set_xlabel('Time since diagnosis (months)'); ax.set_ylabel('Overall survival probability')
ax.set_ylim(0,1.02); ax.legend(loc='lower left', frameon=False, fontsize=9.5)
ax.set_title('Overall Survival by Driver Mutation Status', pad=12)
add_at_risk_counts(*fitters, ax=ax, rows_to_show=['At risk'])
plt.tight_layout(); plt.savefig('outputs/Figure3_KM_by_mutation.png', dpi=300, bbox_inches='tight'); plt.show()

print('\nNo Mutation 대비 쌍별 log-rank:')
for g in ['EGFR','ALK','KRAS','ROS1']:
    a, b = sub[sub.Genetic_Mutation==g], sub[sub.Genetic_Mutation=='No Mutation']
    print(f'  {g}: p={logrank_test(a.Survival_Months,b.Survival_Months,a.event,b.event).p_value:.3f}')

In [ ]:
# STEP 4-2. 변이별 표적치료 수혜율 — 정밀의료 사슬 검증
tt = df.groupby('Genetic_Mutation').apply(
    lambda x: pd.Series({'n':len(x), '표적치료 n':(x.Treatment=='Targeted Therapy').sum(),
                         '표적치료 %':(x.Treatment=='Targeted Therapy').mean()*100}),
    include_groups=False).round(1).sort_values('표적치료 %', ascending=False)
display(tt)

actionable = df.Genetic_Mutation.isin(['EGFR','ALK','ROS1','RET','BRAF','MET','HER2','KRAS'])
p = stats.chi2_contingency(pd.crosstab(actionable, df.Treatment=='Targeted Therapy'))[1]
print(f'Actionable 변이 여부와 표적치료 수혜의 연관: p={p:.3f}')
print('-> p가 유의하지 않다면: 표적치료가 변이와 무관하게 배정되었다는 뜻 (실제 진료와 모순!)')

In [ ]:
# STEP 4-3. EGFR 변이군: 표적치료 vs 기타 치료 (Figure 4) — 교란 검증 포함
eg = df[df.Genetic_Mutation=='EGFR'].copy()
eg['grp'] = np.where(eg.Treatment=='Targeted Therapy','Targeted therapy','Other treatment')

lr = logrank_test(eg[eg.grp=='Targeted therapy'].Survival_Months, eg[eg.grp=='Other treatment'].Survival_Months,
                  eg[eg.grp=='Targeted therapy'].event, eg[eg.grp=='Other treatment'].event)
fig, ax = plt.subplots(figsize=(7.2,6.0), dpi=150)
fitters=[]
for g,c in [('Targeted therapy',OKABE['blue']),('Other treatment',OKABE['vermillion'])]:
    m = eg.grp==g
    k = KaplanMeierFitter().fit(eg.Survival_Months[m], eg.event[m], label=f'{g} (n = {m.sum()})')
    k.plot_survival_function(ax=ax, ci_show=True, ci_alpha=0.12, color=c, linewidth=2)
    fitters.append(k)
    print(f'{g}: n={m.sum()}, median={k.median_survival_time_:.0f}개월')
ax.text(0.97,0.965, f'Log-rank P = {lr.p_value:.3f}', transform=ax.transAxes, ha='right', va='top', fontweight='bold')
ax.set_xlabel('Time since diagnosis (months)'); ax.set_ylabel('Overall survival probability')
ax.set_ylim(0,1.02); ax.legend(loc='lower left', frameon=False)
ax.set_title('EGFR-Mutant Cohort: Targeted Therapy vs Other Treatment', pad=12)
add_at_risk_counts(*fitters, ax=ax, rows_to_show=['At risk'])
plt.tight_layout(); plt.savefig('outputs/Figure4_EGFR_TT_vs_other.png', dpi=300, bbox_inches='tight'); plt.show()

# 교란(적응증) 검증: 두 군의 병기 분포 + 병기·나이 보정 Cox
print('\n두 군의 병기 분포 (행 %):')
display(pd.crosstab(eg.grp, eg.Cancer_Stage, normalize='index').round(2))
egc = eg.copy(); egc['TT'] = (egc.grp=='Targeted therapy').astype(int)
egc = pd.get_dummies(egc[['Survival_Months','event','TT','Cancer_Stage','Age']], columns=['Cancer_Stage'], drop_first=True)
cph_e = CoxPHFitter().fit(egc, 'Survival_Months', 'event')
r = cph_e.summary.loc['TT']
print(f"병기·나이 보정 후 표적치료 HR = {r['exp(coef)']:.2f} (95% CI {r['exp(coef) lower 95%']:.2f}–{r['exp(coef) upper 95%']:.2f}), p={r['p']:.3f}")
print('-> 미보정 유의성이 보정 후 사라지면: 적응증에 의한 교란(confounding by indication)')

## STEP 5. 다변량 Cox 비례위험 모형
NSCLC로 한정(아형 변수가 NSCLC 전용), 희소 변이는 'Other actionable'로 병합, EGFR·Stage I·수술 단독·비흡연을 기준으로 합니다.

In [ ]:
# STEP 5-1. 모형 적합
d = df[df.Cancer_Type=='NSCLC'].copy()
mut_map = {'EGFR':'EGFR','ALK':'ALK','KRAS':'KRAS','ROS1':'ROS1','TP53':'TP53','No Mutation':'No Mutation',
           'BRAF':'Other actionable','MET':'Other actionable','HER2':'Other actionable','RET':'Other actionable'}
d['Mutation'] = d.Genetic_Mutation.map(mut_map)
print(f'NSCLC 코호트: n={len(d)}, 사망 {d.event.sum()}건')

def dz(s, ref, name):
    c = pd.Categorical(s, categories=[ref]+[x for x in s.unique() if x!=ref])
    return pd.get_dummies(pd.Series(c, name=name), prefix=name, drop_first=True)

X = pd.concat([d[['Survival_Months','event','Age']].reset_index(drop=True),
    dz(d.Gender,'Female','Sex').reset_index(drop=True),
    dz(d.Smoking_Status,'Never Smoked','Smoking').reset_index(drop=True),
    dz(d.Cancer_Stage,'Stage I','Stage').reset_index(drop=True),
    dz(d.NSCLC_Subtype,'Adenocarcinoma','Subtype').reset_index(drop=True),
    dz(d.Mutation,'EGFR','Mut').reset_index(drop=True),
    dz(d.Treatment,'Surgery','Tx').reset_index(drop=True),
    dz(d.Metastasis,'No','Metastasis').reset_index(drop=True)], axis=1)

cph = CoxPHFitter().fit(X, 'Survival_Months', 'event')
print(f'C-index = {cph.concordance_index_:.3f}\n')
summ = cph.summary[['exp(coef)','exp(coef) lower 95%','exp(coef) upper 95%','p']].round(3)
summ.columns = ['HR','CI lower','CI upper','p']
summ.to_csv('outputs/Cox_results.csv')
display(summ)

In [ ]:
# STEP 5-2. 비례위험 가정 검정 (Schoenfeld 잔차) + 층화 민감도분석
ph = proportional_hazard_test(cph, X, time_transform='rank')
pv = ph.summary['p'].round(4).sort_values()
print('Schoenfeld p<0.05 (위반):', list(pv[pv<0.05].index) or '없음')
print('(다중검정 고려: 23개 중 1개 경계선 위반은 우연 범위일 수 있음)\n')

# 위반 시 대안 예시: 병기 층화 Cox — 나머지 HR의 안정성 확인
Xs = X.drop(columns=[c for c in X.columns if c.startswith('Stage_')]).copy()
Xs['StageGrp'] = d.Cancer_Stage.values
cph_s = CoxPHFitter().fit(Xs, 'Survival_Months', 'event', strata=['StageGrp'])
print(f'층화 모형 C-index = {cph_s.concordance_index_:.3f} (주 모형과 비교해 병기의 판별 기여도 확인)')
print('\n다른 대안: 시간가변 계수 모형(covariate x log(t)), landmark 분석, AFT 모형')

In [ ]:
# STEP 5-3. Forest plot (Figure 5)
s = pd.read_csv('outputs/Cox_results.csv', index_col=0)
R=[]
def hdr(t): R.append((t,)+(None,)*4+(False,True))
def ref(t): R.append((t,1.0,None,None,None,True,False))
def row(t,k):
    r=s.loc[k]; R.append((t,r['HR'],r['CI lower'],r['CI upper'],r['p'],False,False))
hdr('Age (per year)'); row('  Age','Age')
hdr('Sex'); ref('  Female'); row('  Male','Sex_Male')
hdr('Smoking status'); ref('  Never'); row('  Former','Smoking_Former Smoker'); row('  Current','Smoking_Current Smoker')
hdr('Cancer stage'); ref('  Stage I'); row('  Stage II','Stage_Stage II'); row('  Stage III','Stage_Stage III'); row('  Stage IV','Stage_Stage IV')
hdr('Histologic subtype'); ref('  Adenocarcinoma'); row('  Squamous cell','Subtype_Squamous Cell'); row('  Large cell','Subtype_Large Cell')
hdr('Driver mutation'); ref('  EGFR'); row('  ALK','Mut_ALK'); row('  KRAS','Mut_KRAS'); row('  ROS1','Mut_ROS1'); row('  TP53','Mut_TP53'); row('  Other actionable','Mut_Other actionable'); row('  No mutation','Mut_No Mutation')
hdr('Treatment'); ref('  Surgery alone'); row('  Surgery + chemo','Tx_Surgery + Chemotherapy'); row('  Chemotherapy','Tx_Chemotherapy'); row('  Chemoradiation','Tx_Chemo + Radiation'); row('  Radiotherapy','Tx_Radiotherapy'); row('  Immunotherapy','Tx_Immunotherapy'); row('  Targeted therapy','Tx_Targeted Therapy'); row('  Palliative care','Tx_Palliative Care')
hdr('Distant metastasis'); ref('  Absent'); row('  Present','Metastasis_Yes')

n=len(R); fig, ax = plt.subplots(figsize=(9.2, 0.34*n+1.8), dpi=150)
y=np.arange(n)[::-1]
for yi,(lab,hr,lo,hi,p,is_ref,is_h) in zip(y,R):
    if is_h:
        ax.text(-0.02, yi, lab, transform=ax.get_yaxis_transform(), fontweight='bold', fontsize=10.5, va='center', ha='right'); continue
    ax.text(-0.02, yi, lab, transform=ax.get_yaxis_transform(), fontsize=10, va='center', ha='right')
    if is_ref:
        ax.plot(1, yi, 's', color='#888', ms=5)
        ax.text(1.55, yi, '1.00 (reference)', transform=ax.get_yaxis_transform(), fontsize=9, va='center', color='#666')
    else:
        sig = (lo>1) or (hi<1)
        c = OKABE['vermillion'] if (sig and hr>1) else (OKABE['blue'] if sig else '#333')
        ax.plot([lo,hi],[yi,yi], color=c, lw=1.6); ax.plot(hr, yi, 'o', color=c, ms=5.5)
        ptxt = 'P<0.001' if p<0.001 else f'P={p:.3f}'
        ax.text(1.55, yi, f'{hr:.2f} ({lo:.2f}–{hi:.2f})   {ptxt}', transform=ax.get_yaxis_transform(), fontsize=9, va='center')
ax.axvline(1, color='#999', lw=1, ls='--'); ax.set_xscale('log'); ax.set_xlim(0.3,1500)
ax.set_xticks([0.5,1,2,10,100,1000]); ax.set_xticklabels(['0.5','1','2','10','100','1000']); ax.minorticks_off()
ax.set_ylim(-0.7,n-0.3); ax.set_yticks([])
for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
ax.set_xlabel('Hazard ratio (95% CI), log scale')
ax.set_title(f'Multivariable Cox Model for Overall Mortality\nNSCLC cohort (n = {len(d):,}; {d.event.sum():,} deaths) · C-index = {cph.concordance_index_:.3f}', pad=14)
ax.annotate('HR (95% CI)              P value', xy=(1.55, n-0.15), xycoords=ax.get_yaxis_transform(), fontsize=9, fontweight='bold')
plt.tight_layout(); plt.savefig('outputs/Figure5_forest_cox.png', dpi=300, bbox_inches='tight'); plt.show()

## STEP 6. 결과 파일 다운로드
`outputs/` 폴더의 Table 1, Cox 결과표, 그림 5종(300 DPI)을 ZIP으로 내려받습니다.

In [ ]:
import shutil
shutil.make_archive('lung_cancer_analysis_outputs', 'zip', 'outputs')
from google.colab import files
files.download('lung_cancer_analysis_outputs.zip')
print('다운로드 시작 — outputs 폴더:', os.listdir('outputs'))

---
## 마무리 체크리스트 (비판적 검증)
- [ ] Table 1의 검정법이 정규성 검정 결과와 일치하는가?
- [ ] 병기별 생존 격차의 **크기**가 실제 등록자료(SEER, 국내 암등록)와 비교해 현실적인가?
- [ ] 변이별 표적치료 수혜율이 정밀의료 원칙(변이→표적치료)과 부합하는가?
- [ ] EGFR군의 미보정 vs 보정 결과 차이를 교란으로 설명할 수 있는가?
- [ ] Cox 모형에서 임상적으로 설명 불가능한 유의 결과(예: 완화의료 HR<1)를 그대로 믿지 않았는가?

> 이 다섯 질문에 답하는 과정이 곧 "AI 산출물을 도메인 지식으로 검증하는" 이 워크숍의 핵심 역량입니다.